# Chemeleon2: end-to-end tutorial (VAE → Latent Diffusion → RL/GRPO)

**Last updated:** 2026-02-26  
**Repo:** https://github.com/hspark1212/chemeleon2  
**Docs:** https://hspark1212.github.io/chemeleon2/  
**Paper:** https://arxiv.org/abs/2511.07158

This notebook is designed to be both:

- **A runnable quickstart**: generate a few crystal structures with a pre-trained Chemeleon2 model.
- **A conceptual + practical deep dive**: understand *how* the three-stage pipeline works (VAE, latent diffusion with a DiT denoiser, and RL fine-tuning with GRPO), and how to train/evaluate/customize it.

> **Compute note.** Full training (VAE/LDM/RL) is expensive. This notebook shows the exact commands and configurations you would run, but the *smoke-test* inference sections are sized to run on a single GPU (preferred) or CPU (slower). The docs recommend CUDA for efficient training/inference.


## What is Chemeleon2?

Chemeleon2 implements a **three-stage pipeline** for crystal structure generation:

1. **VAE module**: encode/decode crystal structures into a continuous latent space  
2. **LDM module**: learn a diffusion model in latent space (DiT-based denoiser)  
3. **RL module**: fine-tune the diffusion model with reinforcement learning using **Group Relative Policy Optimization (GRPO)**

This modular pipeline (VAE → LDM → RL) is the main design of Chemeleon2.


## Notebook outline

1. Setup (clone + install)
2. Quickstart: sample a few structures with a pre-trained model
3. Data representation (`CrystalBatch`)
4. VAE theory + how the encoder/decoder are used
5. Latent diffusion theory (DDPM/DDIM, DiT, CFG) + sampling
6. RL/GRPO theory + reward design (creativity/stability/diversity, predictors)
7. Training pipeline (Hydra configs + scripts)
8. Evaluation metrics (uniqueness, novelty, stability, diversity) + mSUN


## 1) Setup

Chemeleon2 targets **Python 3.11+** and recommends using **uv** for reproducible installs (via `uv.lock`). The docs also recommend installing a CUDA-compatible PyTorch build for best performance.

You can run the next cell as-is in a fresh environment. If you already cloned/installed, it should be a no-op.


In [ ]:
import os, sys, pathlib, subprocess, textwrap

REPO_URL = "https://github.com/hspark1212/chemeleon2"
REPO_DIR = pathlib.Path("chemeleon2")

print("Python:", sys.version)
print("Working dir:", os.getcwd())

if not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", REPO_URL])
else:
    print("Repo already exists:", REPO_DIR)

# Install in editable mode.
# If you're using uv (recommended in the docs), you can run:
#   cd chemeleon2 && uv sync
# Here we use pip for maximum compatibility with Jupyter/Colab environments.
print("Installing chemeleon2 (editable) ...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])

# Add repo root to path for direct `src.*` imports if needed
sys.path.insert(0, str(REPO_DIR.resolve()))
print("Done.")


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Working dir: /content
Cloning https://github.com/hspark1212/chemeleon2 ...
Installing chemeleon2 (editable) ...
Done.


### CUDA / PyTorch

If you have a CUDA GPU, install a PyTorch build matching your CUDA runtime (see PyTorch install instructions). The Chemeleon2 installation guide shows an example `uv pip install ... --index-url ...` command for CUDA builds.

In this notebook, we’ll automatically select `"cuda"` if available, otherwise `"cpu"`.


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", device)


torch: 2.10.0+cpu
device: cpu


## 2) Quickstart: sample structures with a pre-trained model

Chemeleon2 provides pre-trained checkpoints via Hugging Face Hub, and the docs show a convenient Python entrypoint:

```python
from src.sample import sample
gen_atoms_list = sample(num_samples=..., batch_size=..., output_dir="...")
```

We’ll generate a **tiny batch** (e.g., 4–8 structures) to keep the smoke test fast.

**Output:** a list of `ase.Atoms` objects.


In [ ]:
from src.sample import sample

# Small smoke test
NUM_SAMPLES = 4
BATCH_SIZE = 4
OUT_DIR = "outputs/quickstart"

gen_atoms_list = sample(
    num_samples=NUM_SAMPLES,
    batch_size=BATCH_SIZE,
    output_dir=OUT_DIR,
)

print("Generated:", len(gen_atoms_list), "structures")
print("Example ASE Atoms:", gen_atoms_list[0])


Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


v0.0.1/alex_mp_20/vae/dng_j1jgz9t0_v1.ck(…):   0%|          | 0.00/205M [00:00<?, ?B/s]

v0.0.1/alex_mp_20/ldm/ldm_rl_dng_tuor5vg(…):   0%|          | 0.00/723M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'encoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['encoder'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'decoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['decoder'])`.


Loaded VAE from checkpoints/v0.0.1/alex_mp_20/vae/dng_j1jgz9t0_v1.ckpt


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'denoiser' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['denoiser'])`.


Loaded model from checkpoints/v0.0.1/alex_mp_20/ldm/ldm_rl_dng_tuor5vgd.ckpt
DNG task: 4 samples
The sampled cif files will be saved in directory: 'outputs/quickstart'
Generating batch #1 with 4 samples.
Using ddim sampler_fn with 50 timesteps.


  0%|          | 0/50 [00:00<?, ?it/s]

The 4 generated structures saved in JSON format at: outputs/quickstart/generated_structures.json.gz
Generated: 4 structures
Example ASE Atoms: MSONAtoms(symbols='Zr2Ti3ZnRe3Rh', pbc=True, cell=[[3.2250771522521973, 0.0, 0.015443701297044504], [0.01169696170836698, 3.223589420318603, 0.03288344293832768], [0.0, 0.0, 15.876649856567383]])


### Visualize one structure (optional)

If you're in a notebook environment, interactive 3D visualization is easiest with `nglview`. If it’s not installed, we’ll just print basic info.

> Tip: `ase.visualize.view(..., viewer="ngl")` is also supported in the docs.


In [ ]:
# Optional: install nglview for inline 3D visualization
# %pip install nglview

from ase import Atoms

atoms0: Atoms = gen_atoms_list[0]
print("Chemical symbols:", atoms0.get_chemical_symbols()[:20], "...")
print("Cell:\n", atoms0.cell)

try:
    import nglview as nv
    view = nv.show_ase(atoms0)
    view
except Exception as e:
    print("NGLView not available (or not supported in this environment).")
    print("Reason:", repr(e))


Chemical symbols: ['Zr', 'Zr', 'Ti', 'Ti', 'Ti', 'Zn', 'Re', 'Re', 'Re', 'Rh'] ...
Cell:
 Cell([[3.2250771522521973, 0.0, 0.015443701297044504], [0.01169696170836698, 3.223589420318603, 0.03288344293832768], [0.0, 0.0, 15.876649856567383]])
NGLView not available (or not supported in this environment).
Reason: ModuleNotFoundError("No module named 'nglview'")


### (Optional) Equivalent CLI sampling

If you prefer the command line (or want an identical run in a batch job), the docs show:

```bash
python src/sample.py --num_samples=100 --batch_size=100 --sampler=ddim --sampling_steps=50 --output_dir=outputs
```

You can also pass explicit checkpoint paths with `--vae_ckpt_path` and `--ldm_ckpt_path`.


In [ ]:
# CLI sampling from inside the notebook (optional).
# Note: running this will write outputs to disk and may download model checkpoints.

import subprocess, sys, pathlib

repo_dir = pathlib.Path("chemeleon2")
cmd = [sys.executable, "src/sample.py", "--num_samples=4", "--batch_size=4", "--output_dir=outputs/cli_quickstart"]

if repo_dir.exists():
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(repo_dir))
else:
    print("Repo directory not found. Run the setup cell first.")


Running: /usr/bin/python3 src/sample.py --num_samples=4 --batch_size=4 --output_dir=outputs/cli_quickstart


### Pre-trained checkpoints (conceptual map)

The quickstart docs list the main checkpoint “identifiers” used throughout configs:

- `mp_20_vae`, `alex_mp_20_vae`
- `mp_20_ldm_base`, `alex_mp_20_ldm_base`
- `mp_20_ldm_rl`, `alex_mp_20_ldm_rl`

In Hydra configs you can refer to these via `${hub:...}` to auto-download from the Hugging Face Hub.


## 3) Under the hood: the three-stage pipeline

Chemeleon2 uses a **latent-space generative pipeline**:

- **Stage A (VAE):** map a crystal structure \(x\) to a latent variable \(z\) (and back).
- **Stage B (LDM):** learn a diffusion model \(p_\theta(z)\) over latents, using a DiT denoiser.
- **Stage C (RL/GRPO):** fine-tune the diffusion model as a *policy* to maximize a reward \(R(x)\) computed from decoded structures.

Why this helps: likelihood-based generation alone tends to over-sample dense regions of known compounds; adding RL enables optimizing explicit, verifiable scientific objectives (novelty, stability, diversity, and property targets).


## 4) Data representation: `CrystalBatch`

Training and sampling operate on a batched tensor representation:

- `atom_types`: atomic numbers (integers)
- `frac_coords`: fractional coordinates \(\in [0,1)^3\)
- `lengths` / `angles`: lattice parameters (cell lengths and angles)
- `num_atoms`: variable atom counts per structure
- `batch`: batch indices (for ragged → batched operations)

This design supports **variable-sized crystals** while still enabling Transformer-style computations (via padding/masks).


In [ ]:
from src.data.schema import create_empty_batch

import torch

# Create an "empty" batch used for generation (num atoms per sample).
# This is the same idea shown in the API docs: create_empty_batch(num_atoms, device=...)
num_atoms = torch.tensor([8, 10, 12], device=device)
batch = create_empty_batch(num_atoms=num_atoms, device=device)

print(batch)
print("atom_types shape:", batch.atom_types.shape)
print("frac_coords shape:", batch.frac_coords.shape)
print("lengths shape:", batch.lengths.shape)
print("angles shape:", batch.angles.shape)
print("num_atoms:", batch.num_atoms)


DataCrystalBatch(pos=[30, 3], atom_types=[30], frac_coords=[30, 3], cart_coords=[30, 3], lattices=[3, 3, 3], num_atoms=[3], lengths=[3, 3], lengths_scaled=[3, 3], angles=[3, 3], angles_radians=[3, 3], token_idx=[30], batch=[30], ptr=[4])
atom_types shape: torch.Size([30])
frac_coords shape: torch.Size([30, 3])
lengths shape: torch.Size([3, 3])
angles shape: torch.Size([3, 3])
num_atoms: tensor([ 8, 10, 12])


## 5) VAE module: theory and architecture

### 5.1 Variational autoencoder objective

A VAE defines:

- an **encoder** \(q_\phi(z\mid x)\)
- a **decoder** \(p_\theta(x\mid z)\)
- a prior \(p(z)\) (often standard normal)

Training minimizes the negative ELBO, which can be written (up to constants) as:

\[
\mathcal{L}_{\text{VAE}}
= \lambda_{\text{recon}}\;\mathcal{L}_{\text{recon}}(x, \hat{x})
+ \lambda_{\text{KL}}\;D_{\mathrm{KL}}\!\left(q_\phi(z\mid x)\,\|\,p(z)\right)
\]

In Chemeleon2, the latent distribution is a diagonal Gaussian parameterized by a mean and log-variance (a `DiagonalGaussianDistribution`), and the encoder/decoder are Transformer-based.


### 5.2 What is being reconstructed?

For a crystal, the decoder predicts enough to reconstruct:

- **atom identities**
- **fractional coordinates**
- **lattice parameters**

Reconstruction losses typically combine terms for each output (classification for species; regression for geometry), while KL regularization encourages a smooth latent space.


### 5.3 Hands-on: featurize a structure with the pre-trained VAE

The `featurize(...)` helper converts `pymatgen.Structure` objects into:

- `structure_features`: latent structure embeddings (VAE latent)
- `composition_features`: composition embeddings
- `atom_features`: per-atom features

We'll build a simple NaCl structure and featurize it.


In [ ]:
from pymatgen.core import Lattice, Structure
from src.utils.featurizer import featurize

# Simple rock-salt NaCl cell (as a toy example)
a = 5.64
latt = Lattice.cubic(a)
species = ["Na", "Cl"]
coords = [
    [0, 0, 0],
    [0.5, 0.5, 0.5],
]
nacl = Structure(latt, species, coords)

features = featurize(
    structures=[nacl],
    model_path=None,   # Uses default pre-trained VAE (downloads if needed)
    batch_size=256,
    device=device,
)

print("Keys:", list(features.keys()))
for k, v in features.items():
    if hasattr(v, "shape"):
        print(k, v.shape)
    else:
        print(k, type(v), "len=", len(v))


v0.0.1/mp_20/vae/dng_m4owq4i5_v0.ckpt:   0%|          | 0.00/205M [00:00<?, ?B/s]

Keys: ['structure_features', 'composition_features', 'atom_features']
structure_features torch.Size([1, 8])
composition_features torch.Size([1, 512])
atom_features <class 'list'> len= 1


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'encoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['encoder'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'decoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['decoder'])`.


## 6) Latent diffusion model (LDM): theory and architecture

Chemeleon2 trains a diffusion model **in the VAE latent space**. Let \(z_0\) be the latent code for a structure.

### 6.1 Forward diffusion (noising)

A standard DDPM forward process adds Gaussian noise:

\[
q(z_t \mid z_{t-1}) = \mathcal{N}\!\left(\sqrt{1-\beta_t}\,z_{t-1},\; \beta_t I\right)
\]

which implies a closed form:

\[
q(z_t \mid z_0) = \mathcal{N}\!\left(\sqrt{\bar{\alpha}_t}\,z_0,\; (1-\bar{\alpha}_t)I\right)
\quad\text{with}\quad
\bar{\alpha}_t = \prod_{s=1}^t (1-\beta_s).
\]

### 6.2 Reverse diffusion (denoising)

The model learns a denoiser (parameterized here by a **Diffusion Transformer / DiT**) to approximate the reverse process, often by predicting noise \(\epsilon_\theta(z_t, t, c)\) given timestep \(t\) and optional condition \(c\).

A common training objective is MSE on noise:

\[
\mathcal{L}_{\text{diff}}
= \mathbb{E}_{t, z_0, \epsilon}\left[\left\|\epsilon - \epsilon_\theta(z_t, t, c)\right\|^2\right].
\]

### 6.3 DiT denoiser + classifier-free guidance (CFG)

Chemeleon2 uses a DiT-style denoiser with timestep/condition embeddings and supports conditional generation (e.g., composition- or property-conditioned). CFG mixes conditional and unconditional predictions:

\[
\epsilon_{\text{CFG}} = (1+w)\,\epsilon_\theta(z_t,t,c) - w\,\epsilon_\theta(z_t,t,\varnothing)
\]

where \(w\) is `cfg_scale`.


### 6.4 Sampling with the trained LDM

There are two typical ways to sample:

- **High-level helper:** `src.sample.sample(...)` (used above)
- **Lower-level API:** load `LDMModule` and call `ldm.sample(batch, sampling_steps=..., sampler="ddim"/"ddpm")`

Below is a lower-level sketch that matches the API reference.  
(You need valid checkpoint paths; the CLI and Hydra configs can auto-download checkpoints via `${hub:...}`.)


In [ ]:
# This cell is a reference implementation pattern.
# It is written to be runnable once you provide valid checkpoint paths (or use the Hydra `${hub:...}` resolver in CLI).

from src.ldm_module import LDMModule
from src.data.schema import create_empty_batch
import torch

# TODO: set these to actual paths on your machine (or download from Hugging Face Hub).
vae_ckpt_path = None  # e.g., "checkpoints/v0.0.1/alex_mp_20/vae/....ckpt"
ldm_ckpt_path = None  # e.g., "checkpoints/v0.0.1/alex_mp_20/ldm/....ckpt"

if vae_ckpt_path and ldm_ckpt_path:
    ldm = LDMModule.load_from_checkpoint(
        ldm_ckpt_path,
        vae_ckpt_path=vae_ckpt_path,
        weights_only=False,
        map_location=device,
    ).to(device)
    ldm.eval()

    num_atoms = torch.tensor([10, 12, 15], device=device)
    batch0 = create_empty_batch(num_atoms=num_atoms, device=device)

    batch_gen = ldm.sample(batch0, sampling_steps=50, sampler="ddim", cfg_scale=2.0)
    structures = batch_gen.to_structure()
    print("Generated", len(structures), "pymatgen structures")
else:
    print("Set `vae_ckpt_path` and `ldm_ckpt_path` to run this cell.")


ImportError: cannot import name 'LDMModule' from 'src.ldm_module' (/content/chemeleon2/src/ldm_module/__init__.py)

## 7) RL fine-tuning with GRPO: theory and architecture

Chemeleon2 treats the diffusion model as a **policy** and fine-tunes it with reinforcement learning to maximize a reward computed from decoded structures.

### 7.1 GRPO objective (PPO-style)

Chemeleon2 uses **Group Relative Policy Optimization (GRPO)**, a PPO-style clipped objective with KL and entropy regularization:

\[
\mathcal{L}_{\mathrm{GRPO}}
=
-\mathbb{E}\left[\min\left(r_t(\theta)A_t,\;\mathrm{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)A_t\right)\right]
+\beta\,D_{\mathrm{KL}}-\gamma\,H
\]

where:

- \(r_t(\theta)\): probability ratio between new and old policies
- \(A_t\): advantage (in GRPO, often computed from *group-normalized* rewards)
- \(\epsilon\): clip parameter
- \(\beta\): KL penalty coefficient
- \(\gamma\): entropy coefficient
- \(H\): entropy bonus to encourage exploration/diversity

### 7.2 Rewards: verifiable, modular components

Rewards are built by composing `RewardComponent`s (e.g., novelty/uniqueness, energy-above-hull, diversity via MMD). Chemeleon2 aggregates them with `ReinforceReward`, supporting per-component normalization and global normalization.


### 7.3 Hands-on: a simple *custom reward* (atomic density)

A great starting point is a reward that **does not require external reference datasets**.  
Atomic density is one such example: \( \rho = \frac{N}{V} \) (atoms per Å³), where \(N\) is the number of atoms and \(V\) is the cell volume.

Below we compute this reward for our sampled structures.


In [ ]:
import numpy as np

def atomic_density_reward(atoms):
    # atoms: ase.Atoms
    N = len(atoms)
    V = atoms.get_volume()  # Å^3
    return N / max(V, 1e-12)

densities = np.array([atomic_density_reward(a) for a in gen_atoms_list])
print("Atomic density (atoms/Å^3):", densities)
print("mean:", densities.mean(), "std:", densities.std())


### 7.4 Property optimization with predictors (optional)

Chemeleon2 supports training a **predictor in VAE latent space** (e.g., for band gap), and then using it as a reward in RL training.

High level workflow:

1. Prepare a dataset with property labels (CSV files containing `cif` + property columns)
2. Train a predictor (`src/train_predictor.py`)
3. Configure RL reward to include predictor-based reward (`PredictorReward` or custom component)
4. Run RL fine-tuning (`src/train_rl.py`)

We do not run predictor training in this notebook (it requires the dataset and significant compute), but we include the exact commands and key configuration concepts.


In [ ]:
# Predictor training (from docs):
#   python src/train_predictor.py custom_reward=predictor_band_gap
#
# RL training with predictor reward (from docs):
#   python src/train_rl.py custom_reward=rl_bandgap
#
# Important: your target property name must match:
#  - the column name in your CSV
#  - the predictor module's target_conditions key
#  - the reward config's target_name
#
# See docs for details.
print("See the 'Predictor-Based Reward' section of the docs for full configs and examples.")


## 8) Training pipeline (VAE → LDM → RL) with Hydra

Chemeleon2 uses **Hydra** config composition. Training scripts:

- `python src/train_vae.py ...`
- `python src/train_ldm.py ...`
- `python src/train_rl.py ...`
- `python src/train_predictor.py ...` (optional)

You typically select an experiment config, e.g.:

```bash
python src/train_vae.py experiment=mp_20/vae_dng
python src/train_ldm.py experiment=mp_20/ldm_null
python src/train_rl.py  experiment=mp_20/rl_dng
```

### Inspecting the resolved config

Hydra lets you print the final merged config without running training:

```bash
python src/train_ldm.py experiment=mp_20/ldm_null --cfg job
```

### Checkpoints

The training guide describes two ways to set checkpoint paths:

- **Auto-download from Hugging Face Hub** using `${hub:...}` in configs
- **Local paths** to `.ckpt` files

Checkpoints are typically saved under:

```
logs/{task}/runs/{timestamp}/checkpoints/
```


In [ ]:
# This cell just prints a helpful reminder; training is typically run from a terminal.
print("Training is usually run from a terminal (or a long-running job) rather than inside a notebook.")
print("Use `python src/train_*.py ...` with Hydra configs as shown above.")


## 9) Evaluation: uniqueness, novelty, stability, diversity (and mSUN)

Chemeleon2 ships an evaluation helper `Metrics` and a CLI script `src/evaluate.py`.

Common metrics include:

- **Unique**: deduplicated structures within your generated set  
- **Novel**: not present in the reference dataset  
- **E above hull**: thermodynamic stability via phase diagrams  
- **Composition validity**: chemical validity (SMACT)  
- **Structure/Composition diversity**: distance between generated and reference embeddings (inverse Fréchet distance)

A commonly reported score is **mSUN** (metastable Structure Uniqueness and Novelty), computed as:

\[
\mathrm{mSUN} = \frac{1}{N}\sum_i \mathbb{1}[\text{unique}_i \land \text{novel}_i \land \text{metastable}_i]
\]

### Reference assets

Some metrics require benchmark assets (reference embeddings/structures/phase diagrams).  
The evaluation guide provides a `curl` command to download a tarball from Figshare.


In [ ]:
# OPTIONAL (recommended for full metrics):
# Download benchmark assets (from docs):
#
#   curl -L -A "Mozilla/5.0" -o benchmarks_mp_20.tar.gz https://figshare.com/ndownloader/files/59462369
#   tar -zxvf benchmarks_mp_20.tar.gz
#
# Then you can evaluate generated structures.
#
# Programmatic evaluation example:

from src.utils.metrics import Metrics
from pymatgen.core import Structure

# Convert our tiny sample from ASE Atoms to pymatgen Structure
gen_structures = [Structure.from_ase_atoms(a) for a in gen_atoms_list]

metrics = Metrics(
    metrics=["unique", "novel"],   # keep light; add "e_above_hull" once assets are downloaded
    reference_dataset="mp-20",
    phase_diagram="mp-all",
    metastable_threshold=0.1,
    progress_bar=True,
)

try:
    results = metrics.compute(gen_structures=gen_structures)
    print("Computed metrics keys:", list(results.keys()))
    print("Unique rate:", results["unique"].mean())
    print("Novel rate:", results["novel"].mean())
except Exception as e:
    print("Metrics computation failed (likely missing benchmark assets).")
    print("Reason:", repr(e))


## 10) Next steps

- **Try different sampling setups**: DDIM vs DDPM, more steps, CFG scale
- **Conditioned generation**: composition-conditioned sampling via `src/sample.py --compositions="LiFePO4,..."`  
- **Customize rewards**: mix creativity + stability + diversity, or add predictor-based rewards
- **Run RL fine-tuning** for a targeted discovery objective

Useful references:

- Chemeleon2 docs home: https://hspark1212.github.io/chemeleon2/
- Training overview: https://hspark1212.github.io/chemeleon2/index-1/
- Architecture overview: https://hspark1212.github.io/chemeleon2/index-3/
- Custom rewards overview: https://hspark1212.github.io/chemeleon2/index-2/
- Evaluation guide: https://hspark1212.github.io/chemeleon2/evaluation/
- Paper: https://arxiv.org/abs/2511.07158
